# Lesson 7 — Train the three VAD models

We're going to train MLP, CNN, and CRNN sequentially, then compare their validation F1.

**Time estimate on L40S** (with epoch-size=5000, 3 epochs each):
- MLP:  ~3 minutes
- CNN:  ~6 minutes
- CRNN: ~12 minutes

Total: ~20-25 minutes for all three. Long enough to step away for coffee, short enough to iterate fast.

For a real production run you'd push to `--epoch-size 50000 --epochs 20`, which takes hours but produces a serious model. We'll do that in the next lesson once we've confirmed everything works end-to-end at small scale.


## 1. Quick training run — MLP baseline first

Always run the baseline first. If the MLP doesn't get to ~80% F1, something is wrong with the data pipeline and the fancy models will fail in mysterious ways.

We use `subprocess` to call `train.py` from the notebook so you see real-time output.


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "/workspace/train.py",
    "--model", "mlp",
    "--epochs", "3",
    "--epoch-size", "5000",
    "--batch-size", "32",
    "--num-workers", "8",
    "--out-dir", "/workspace/checkpoints",
]
# Stream output line by line so you see progress live
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f"\nExit code: {proc.returncode}")


**What to expect for the MLP:**

- Train loss starts around 0.69 (random predictions), drops to roughly 0.3-0.4 by end
- Validation F1 climbs from ~0.5 to ~0.80-0.85
- Best checkpoint saved to `/workspace/checkpoints/best_mlp.pt`

If F1 stalls below 0.7, common causes are:
- Data loader is broken (re-run lesson 5 checks)
- Learning rate too high → check loss isn't oscillating wildly
- Labels miscomputed → check the speech_frac in the logs is reasonable (~0.6-0.8)

---

## 2. Train the CNN

This is where temporal context kicks in. Expect F1 to jump 5-10 points over the MLP.


In [ ]:
cmd = [
    sys.executable, "/workspace/train.py",
    "--model", "cnn",
    "--epochs", "3",
    "--epoch-size", "5000",
    "--batch-size", "32",
    "--num-workers", "8",
    "--out-dir", "/workspace/checkpoints",
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f"\nExit code: {proc.returncode}")


**What to expect for the CNN:**

- Final F1 around 0.90-0.93
- Clear improvement over MLP — this is the proof that temporal context matters
- If it doesn't beat the MLP, something is wrong (most likely the model isn't seeing enough context, or training data has labels that don't reward context — neither of which should happen with our setup)

---

## 3. Train the CRNN

The full model. The GRU adds long-range memory on top of the CNN's local context.


In [ ]:
cmd = [
    sys.executable, "/workspace/train.py",
    "--model", "crnn",
    "--epochs", "3",
    "--epoch-size", "5000",
    "--batch-size", "32",
    "--num-workers", "8",
    "--out-dir", "/workspace/checkpoints",
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f"\nExit code: {proc.returncode}")


**What to expect for the CRNN:**

- Final F1 around 0.92-0.95
- Modest improvement over CNN (1-3 points). The big jump is MLP→CNN; CNN→CRNN is smaller because the CNN already captures most of the value.
- At small training scale, the CRNN's advantage is muted. At large scale (50K+ samples/epoch), the gap widens because the recurrent layer needs more data to fully exploit.

If the CRNN is *worse* than the CNN at small scale, that's actually normal — more parameters need more data. The fix is to either train longer or shrink the GRU.

---

## 4. Compare all three side-by-side

Load each checkpoint and run a fixed validation set through them. Plot side-by-side predictions on the same input.


In [ ]:
import sys
sys.path.insert(0, '/workspace')

import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

from vad_dataset import VADDataset
from vad_models import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build a deterministic small validation set (same speakers as during training)
from train import split_speakers, _SpeakerFilteredDataset
_, val_speakers = split_speakers("/workspace/data/LibriSpeech/train-clean-100",
                                  val_speaker_count=10)
val_ds = _SpeakerFilteredDataset(
    allowed_speakers=val_speakers,
    speech_dir="/workspace/data/LibriSpeech/train-clean-100",
    noise_dir="/workspace/data/musan/noise",
    music_dir="/workspace/data/musan/music",
    epoch_size=200,
    seed=99,  # fixed so we always look at the same samples
)

# Load the trained models
models = {}
for name in ['mlp', 'cnn', 'crnn']:
    ckpt = torch.load(f"/workspace/checkpoints/best_{name}.pt", map_location=device)
    m = build_model(name).to(device)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    models[name] = m
    print(f"Loaded {name.upper():5}: trained for {ckpt['step']} steps, val F1 {ckpt['f1']:.3f}")


In [ ]:
# Plot predictions from each model on the same 3 samples
fig, axes = plt.subplots(3, 5, figsize=(22, 9))

for row in range(3):
    feat, lab = val_ds[row]
    feat_gpu = feat.unsqueeze(0).to(device)
    times = np.arange(lab.shape[0]) * 160 / 16000

    # Spectrogram
    librosa.display.specshow(feat.numpy(), sr=16000, hop_length=160,
                             x_axis='time', y_axis='mel', cmap='magma', ax=axes[row, 0])
    axes[row, 0].set_title(f"Sample {row+1}: log-mel input")

    # Ground truth
    axes[row, 1].fill_between(times, 0, lab.numpy(), step='mid', alpha=0.6, color='C2')
    axes[row, 1].set_ylim(-0.1, 1.1); axes[row, 1].set_title("Ground truth")

    # Each model
    for col, (name, m) in enumerate(models.items()):
        with torch.no_grad():
            probs = torch.sigmoid(m(feat_gpu)).squeeze().cpu().numpy()
        ax = axes[row, 2 + col]
        ax.plot(times, probs, color='C1', linewidth=1)
        ax.fill_between(times, 0, lab.numpy(), step='mid', alpha=0.2, color='C2')
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.5)
        ax.set_ylim(-0.1, 1.1)
        ax.set_title(f"{name.upper()} predictions")

plt.tight_layout()
plt.show()


**Reading this plot:**

For each sample (row), columns 2-5 show: ground truth (green bar), then MLP, CNN, CRNN predictions (orange line) with the ground truth as a faded green underlay.

What you want to see:

- **MLP**: noisy predictions that mostly track the ground truth but flicker a lot — predictions jump between 0.2 and 0.8 frame-to-frame even within a single word
- **CNN**: smoother predictions, fewer false drops within words, cleaner transitions at word boundaries
- **CRNN**: smoothest, with clear plateau regions during speech and silence

The visual improvement is striking once you see it. This is what temporal context buys you.

---

## 5. Quantitative comparison

A single number to summarize the difference.


In [ ]:
from train import validate
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([0.5], device=device))

# Bigger val set for proper measurement
val_ds_big = _SpeakerFilteredDataset(
    allowed_speakers=val_speakers,
    speech_dir="/workspace/data/LibriSpeech/train-clean-100",
    noise_dir="/workspace/data/musan/noise",
    music_dir="/workspace/data/musan/music",
    epoch_size=2000,
    seed=12345,
)
val_loader_big = torch.utils.data.DataLoader(
    val_ds_big, batch_size=32, num_workers=4, pin_memory=True,
)

print(f"{'Model':6} | {'val_loss':>9} | {'precision':>10} | {'recall':>8} | {'F1':>6}")
print("-" * 50)
for name, m in models.items():
    val_loss, p, r, f1 = validate(m, val_loader_big, loss_fn, device, max_batches=60)
    print(f"{name.upper():6} | {val_loss:>9.4f} | {p:>10.3f} | {r:>8.3f} | {f1:>6.3f}")


**Healthy result looks like:**

```
Model  | val_loss  | precision |   recall |     F1
--------------------------------------------------
MLP    |   0.350   |     0.880 |   0.815  |  0.846
CNN    |   0.215   |     0.918 |   0.901  |  0.910
CRNN   |   0.180   |     0.935 |   0.918  |  0.926
```

Numbers vary based on training randomness, but the ordering MLP < CNN < CRNN should hold. If it doesn't, ping me with what you got.

---

## You're done with lesson 7 when…

- All three models trained successfully (no NaN, no errors)
- Final F1 ordering is MLP < CNN < CRNN
- You have three checkpoints in `/workspace/checkpoints/`
- The qualitative plot makes the difference between architectures obvious

This is genuinely a **working VAD**. Not yet competitive with Silero, but it works. To make it competitive we need:

- More training data (DNS Challenge)
- Longer training (50K+ epoch size, 20+ epochs)
- Better labels (distillation from a pretrained VAD)
- Augmentation tricks (reverberation, speech speed perturbation, gain perturbation)

That's lesson 8. But first — celebrate. You went from "I don't know what a VAD is" to "I have three trained neural VADs" in seven lessons.

Ping me with one of:
- **"All trained, take me to lesson 8 (closing the gap to Silero)"**
- **"Training failed on model X — here's the output"**
- **"My F1 ordering is weird — here are the numbers"**
- **"How do I actually USE this VAD on a new audio file?"** → I'll write a quick inference script
